In [ ]:
pip install squarify

Dataset Structure Analyzer

1: Size Formatting Function

In [ ]:
from datetime import datetime

def format_size(size_bytes):
    """Convert bytes to human-readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.1f}{unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.1f}PB"

2.File Metadata Functions

In [ ]:
import os

def get_file_metadata(file_path):
    """Get generalized metadata for a file"""
    try:
        stats = os.stat(file_path)
        size = format_size(stats.st_size)
        mod_time = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d')
        ext = os.path.splitext(file_path)[1].upper().replace('.', '') or 'FILE'
        return f"{{Size: {size}, Type: {ext}, Modified: {mod_time}}}"
    except:
        return "{Size: N/A}"

3: ZIP Item Metadata Function

In [ ]:
def get_zip_item_metadata(zip_ref, item_name):
    """Get metadata for items inside ZIP files"""
    try:
        info = zip_ref.getinfo(item_name)
        size = format_size(info.file_size)
        
        # Handle modification date
        try:
            mod_time = datetime(*info.date_time).strftime('%Y-%m-%d')
        except:
            mod_time = 'N/A'
        
        ext = os.path.splitext(item_name)[1].upper().replace('.', '') or 'FILE'
        return f"{{Size: {size}, Type: {ext}, Modified: {mod_time}}}"
    except Exception as e:
        return "{Size: N/A, Type: N/A, Modified: N/A}"

4: Tree Generator Function

In [ ]:
import zipfile
from io import BytesIO

# Global variable to store output
tree_output = ""

def print_tree(max_depth=None, show_zip_contents=True, save_to_string=False):
    """
    Print a clean directory tree structure with ZIP file exploration
    Uses the 'path' variable defined in your code
    
    Args:
        max_depth: Maximum depth to traverse (None for unlimited)
        show_zip_contents: If True, shows contents of ZIP files without extracting
        save_to_string: If True, stores output in global variable instead of printing
    """
    global tree_output
    tree_output = ""  # Reset output
    
    def _output(text):
        """Helper to either print or store output"""
        global tree_output
        if save_to_string:
            tree_output += text + "\n"
        else:
            print(text)
    
    def _explore_zip(zip_path, prefix="", depth=0, parent_zip=None, is_root=False):
        """Explore ZIP file contents without extraction"""
        try:
            # If this is a nested ZIP, read it from the parent ZIP
            if parent_zip:
                zip_data = BytesIO(parent_zip.read(zip_path))
                zip_ref = zipfile.ZipFile(zip_data, 'r')
            else:
                zip_ref = zipfile.ZipFile(zip_path, 'r')
            
            with zip_ref:
                # Get all file names in the zip, filter out __MACOSX and .DS_Store
                zip_contents = [item for item in zip_ref.namelist() 
                               if not item.startswith('__MACOSX/') 
                               and item != '__MACOSX'
                               and not item.endswith('.DS_Store')
                               and item != '.DS_Store']
                
                # Build a tree structure from flat paths
                tree_dict = {}
                zip_files = {}  # Track which items are ZIP files
                zip_info = {}   # Store ZipInfo objects for metadata - maps display path to original path
                
                for item in zip_contents:
                    parts = item.split('/')
                    current = tree_dict
                    for j, part in enumerate(parts):
                        if part:  # Skip empty strings
                            if part not in current:
                                current[part] = {}
                            current = current[part]
                            # Mark if this is a ZIP file
                            if part.lower().endswith('.zip') and j == len(parts) - 1:
                                zip_files[item] = True
                    
                    # Store original path for files (not directories)
                    if not item.endswith('/'):
                        zip_info[item] = item  # Map original path to itself initially
                
                # Determine root path to skip
                root_to_skip = None
                if is_root and len(tree_dict) == 1:
                    top_key = list(tree_dict.keys())[0]
                    # Check if it's a directory (has children)
                    if len(tree_dict[top_key]) > 0:
                        root_to_skip = top_key + '/'
                        tree_dict = tree_dict[top_key]
                
                # Print the tree structure
                _print_zip_tree(tree_dict, prefix, depth, zip_ref, zip_files, 
                              zip_info=zip_info, root_to_skip=root_to_skip)
                
        except (zipfile.BadZipFile, PermissionError) as e:
            _output(f"{prefix}    [Cannot read ZIP: {str(e)}]")
    
    def _print_zip_tree(tree_dict, prefix="", depth=0, parent_zip=None, zip_files=None, 
                       current_path="", zip_info=None, root_to_skip=None):
        """Print tree structure from nested dictionary"""
        items = sorted(tree_dict.items())
        zip_files = zip_files or {}
        zip_info = zip_info or {}
        
        for i, (name, subtree) in enumerate(items):
            is_last = i == len(items) - 1
            connector = "└── " if is_last else "├── "
            
            # Build display path
            display_path = f"{current_path}/{name}" if current_path else name
            
            # Build original path (with root if it exists)
            if root_to_skip:
                original_path = root_to_skip + display_path
            else:
                original_path = display_path
            
            # Check if it's a directory (has children) or file (empty dict)
            is_dir = len(subtree) > 0
            is_zip = name.lower().endswith('.zip')
            
            # Choose indicator
            if is_zip and show_zip_contents:
                indicator = "🗜️ "
            elif is_dir:
                indicator = "📁 "
            else:
                indicator = ""
            
            # Add metadata for files
            if not is_dir and parent_zip:
                try:
                    # Try to get info using original path
                    info = parent_zip.getinfo(original_path)
                    size = format_size(info.file_size)
                    try:
                        mod_time = datetime(*info.date_time).strftime('%Y-%m-%d')
                    except:
                        mod_time = 'N/A'
                
                    metadata = f"{{Size: {size}, Modified: {mod_time}}}"
                except:
                    metadata = "{Size: N/A, Type: N/A, Modified: N/A}"
                _output(f"{prefix}{connector}{indicator}{name} {metadata}")
            else:
                _output(f"{prefix}{connector}{indicator}{name}")
            
            # Explore nested ZIP files
            if is_zip and show_zip_contents and parent_zip:
                extension = "    " if is_last else "│   "
                _explore_zip(original_path, prefix + extension, depth + 1, parent_zip, is_root=False)
            # Recurse if it's a directory
            elif is_dir:
                extension = "    " if is_last else "│   "
                _print_zip_tree(subtree, prefix + extension, depth + 1, parent_zip, 
                              zip_files, display_path, zip_info, root_to_skip)
    
    def _tree_generator(dir_path, prefix="", depth=0):
        """Recursively generate tree structure"""
        
        # Check depth limit
        if max_depth is not None and depth >= max_depth:
            return
        
        try:
            # Get and sort contents
            contents = sorted(os.listdir(dir_path))
            
            # Separate directories and files, filter out unwanted items
            dirs = [item for item in contents 
                   if os.path.isdir(os.path.join(dir_path, item)) 
                   and item != '__pycache__'
                   and not item.startswith('._')]
            files = [item for item in contents 
                    if os.path.isfile(os.path.join(dir_path, item)) 
                    and item != '.DS_Store'
                    and not item.startswith('._')]
            
            # Combine: directories first, then files
            all_items = dirs + files
            
            for i, item in enumerate(all_items):
                item_path = os.path.join(dir_path, item)
                is_last = i == len(all_items) - 1
                is_dir = item in dirs  # Check if it's in dirs list
                
                # Choose connector
                connector = "└── " if is_last else "├── "
                
                # Check if it's a ZIP file
                is_zip = item.lower().endswith('.zip')
                
                # Print item with optional indicator
                if is_zip and show_zip_contents:
                    indicator = "🗜️ "
                else:
                    indicator = "📁 " if is_dir else ""
                
                # Add metadata for files (not directories)
                if not is_dir:
                    metadata = get_file_metadata(item_path)
                    _output(f"{prefix}{connector}{indicator}{item} {metadata}")
                else:
                    _output(f"{prefix}{connector}{indicator}{item}")
                
                # Handle ZIP files
                if is_zip and show_zip_contents:
                    extension = "    " if is_last else "│   "
                    _explore_zip(item_path, prefix + extension, depth + 1)
                # Recurse into directories
                elif is_dir:
                    extension = "    " if is_last else "│   "
                    _tree_generator(item_path, prefix + extension, depth + 1)
                    
        except PermissionError:
            _output(f"{prefix}[Permission Denied]")
    
    # Print root
    root_name = os.path.basename(os.path.abspath(path)) or path
    
    # Check if the path itself is a ZIP file
    if os.path.isfile(path) and path.lower().endswith('.zip'):
        # Remove .zip extension for display
        display_name = root_name[:-4] if root_name.endswith('.zip') else root_name
        _output(f"📂 {display_name}")
        if show_zip_contents:
            _explore_zip(path, "", 0, is_root=True)
    else:
        _output(f"📂 {root_name}")
        _tree_generator(path)

5: Extension Counter Function

In [ ]:
from collections import defaultdict

# Global variables to store extension counts and sizes
extension_counts = defaultdict(int)
extension_sizes = defaultdict(int)  # Store cumulative size in bytes

def count_extensions(show_zip_contents=True):
    """
    Count file extensions and sizes in directory/ZIP structure
    Uses the 'path' variable defined in your code
    
    Args:
        show_zip_contents: If True, counts files inside ZIP files too
    """
    global extension_counts, extension_sizes
    extension_counts = defaultdict(int)  # Reset counts
    extension_sizes = defaultdict(int)  # Reset sizes
    
    # Track if we're processing the root ZIP to avoid counting it
    root_zip_path = os.path.abspath(path) if os.path.isfile(path) and path.lower().endswith('.zip') else None
    
    def _count_in_zip(zip_path, parent_zip=None, is_root=False):
        """Count extensions in ZIP file contents"""
        try:
            if parent_zip:
                zip_data = BytesIO(parent_zip.read(zip_path))
                zip_ref = zipfile.ZipFile(zip_data, 'r')
            else:
                zip_ref = zipfile.ZipFile(zip_path, 'r')
            
            with zip_ref:
                zip_contents = [item for item in zip_ref.namelist() 
                               if not item.startswith('__MACOSX/') 
                               and not item.endswith('.DS_Store')
                               and not item.startswith('._')]
                
                for item in zip_contents:
                    # Skip directories
                    if item.endswith('/'):
                        continue
                    
                    # Get extension and file info
                    _, ext = os.path.splitext(item)
                    if ext:
                        # Skip counting ZIP files since we're exploring their contents
                        if ext.lower() == '.zip' and show_zip_contents:
                            # Explore the nested ZIP but don't count it
                            _count_in_zip(item, zip_ref, is_root=False)
                        else:
                            # Count non-ZIP files
                            extension_counts[ext.lower()] += 1
                            # Get file size from ZIP - use compress_size or file_size
                            try:
                                file_info = zip_ref.getinfo(item)
                                # Use file_size (uncompressed), fallback to compress_size
                                size = file_info.file_size if file_info.file_size > 0 else file_info.compress_size
                                extension_sizes[ext.lower()] += size
                            except:
                                pass
                        
        except (zipfile.BadZipFile, PermissionError):
            pass
    
    def _count_in_directory(dir_path):
        """Recursively count extensions in directory"""
        try:
            contents = os.listdir(dir_path)
            
            for item in contents:
                if item.startswith('._') or item in ['__pycache__', '.DS_Store']:
                    continue
                
                item_path = os.path.join(dir_path, item)
                
                if os.path.isfile(item_path):
                    # Skip if this is the root ZIP file we're analyzing
                    if root_zip_path and os.path.abspath(item_path) == root_zip_path:
                        continue
                    
                    _, ext = os.path.splitext(item)
                    if ext:
                        # If it's a ZIP file and we're exploring contents, don't count the ZIP itself
                        if ext.lower() == '.zip' and show_zip_contents:
                            # Explore it but don't count it
                            _count_in_zip(item_path, is_root=False)
                        else:
                            # Count non-ZIP files
                            extension_counts[ext.lower()] += 1
                            # Get file size
                            try:
                                extension_sizes[ext.lower()] += os.path.getsize(item_path)
                            except:
                                pass
                
                elif os.path.isdir(item_path):
                    _count_in_directory(item_path)
                    
        except PermissionError:
            pass
    
    # Start counting
    if os.path.isfile(path) and path.lower().endswith('.zip'):
        # If the path itself is a ZIP, explore it without counting the ZIP file itself
        _count_in_zip(path, is_root=True)
    else:
        _count_in_directory(path)
    
    return extension_counts

6: Extension Summary Printer Function

In [ ]:
def print_extension_summary():
    """Print a formatted table of extension counts and sizes"""
    if not extension_counts:
        print("No files found or extensions counted.")
        return
    
    def format_size_summary(size_bytes):
        """Convert bytes to human-readable format"""
        if size_bytes == 0:
            return "0 B"
        for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
            if size_bytes < 1024.0:
                return f"{size_bytes:.2f} {unit}"
            size_bytes /= 1024.0
        return f"{size_bytes:.2f} PB"
    
    # Sort by count (descending) then by extension name
    sorted_exts = sorted(extension_counts.items(), key=lambda x: (-x[1], x[0]))
    
    # Calculate totals
    total_files = sum(extension_counts.values())
    total_size = sum(extension_sizes.values())
    
    # Print header
    print("\n📊 Extension Summary")
    print("=" * 80)
    print(f"{'Extension':<15} {'Count':>10} {'Percentage':>12} {'Total Size':>15}")
    print("-" * 80)
    
    # Print each extension
    for ext, count in sorted_exts:
        percentage = (count / total_files) * 100
        size = extension_sizes[ext]
        print(f"{ext:<15} {count:>10,} {percentage:>11.1f}% {format_size_summary(size):>15}")
    
    # Print total
    print("-" * 80)
    print(f"{'TOTAL':<15} {total_files:>10,} {100.0:>11.1f}% {format_size_summary(total_size):>15}")
    print("=" * 80)

7: File Saver Function

In [ ]:
def save_tree_to_file(filename, output_string, include_summary=True):
    """
    Save the tree structure output to a text file
    Args:
        filename: Name of the file to save (e.g., 'directory_structure.txt')
        output_string: The tree output string to save
        include_summary: If True, includes extension summary in the file
    """
    try:
        # Get the directory of the path variable
        base_dir = os.path.dirname(path) if os.path.isfile(path) else os.path.dirname(path.rstrip(os.sep))
        file_path = os.path.join(base_dir, filename)
        
        # Write the output to file
        with open(file_path, 'w', encoding='utf-8') as f:
            # Add extension summary first if requested
            if include_summary and extension_counts:
                def format_size(size_bytes):
                    """Convert bytes to human-readable format"""
                    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
                        if size_bytes < 1024.0:
                            return f"{size_bytes:.2f} {unit}"
                        size_bytes /= 1024.0
                    return f"{size_bytes:.2f} PB"
                
                sorted_exts = sorted(extension_counts.items(), 
                                    key=lambda x: (-x[1], x[0]))
                total_files = sum(extension_counts.values())
                total_size = sum(extension_sizes.values())
                
                f.write("📊 Extension Summary\n")
                f.write("=" * 80 + "\n")
                f.write(f"{'Extension':<15} {'Count':>10} {'Percentage':>12} {'Total Size':>15}\n")
                f.write("-" * 80 + "\n")
                
                for ext, count in sorted_exts:
                    percentage = (count / total_files) * 100
                    size = extension_sizes[ext]
                    f.write(f"{ext:<15} {count:>10,} {percentage:>11.1f}% {format_size(size):>15}\n")
                
                f.write("-" * 80 + "\n")
                f.write(f"{'TOTAL':<15} {total_files:>10,} {100.0:>11.1f}% {format_size(total_size):>15}\n")
                f.write("=" * 80 + "\n\n")
            
            # Then add directory structure
            f.write("Directory Structure:\n")
            f.write("=" * 60 + "\n")
            f.write(output_string)
            f.write("=" * 60 + "\n")
        
        print(f"✅ Tree structure saved successfully to: {file_path}")
        return file_path
        
    except Exception as e:
        print(f"❌ Error saving file: {str(e)}")
        return None

FOR SINGLE DATASET

In [ ]:
# Set your dataset path (directory or ZIP file)
path = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1"

# Generate tree structure
print("Generating directory tree...")
print_tree(save_to_string=True)

# Count extensions
print("\nCounting file extensions...")
count_extensions(show_zip_contents=True)

# Display extension summary
print_extension_summary()

# Save to file
print("\nSaving to file...")
base_name = os.path.splitext(os.path.basename(path))[0]
output_filename = f"directory_tree_{base_name}.txt"
saved_path = save_tree_to_file(output_filename, tree_output, include_summary=True)

print("\n✅ Analysis complete!")

Batch Processing Function

In [ ]:
from pathlib import Path

def process_batch_datasets(parent_path, show_zip_contents=True):
    """
    Process all datasets one level below the parent path
    
    Args:
        parent_path: Path to directory containing datasets
        show_zip_contents: If True, explores ZIP contents
    """
    global path, tree_output, extension_counts, extension_sizes
    
    try:
        # Get all items one level below
        items = os.listdir(parent_path)
        
        # Filter: directories or ZIP files only
        datasets = [item for item in items 
                   if (os.path.isdir(os.path.join(parent_path, item)) or 
                       item.lower().endswith('.zip'))
                   and not item.startswith('.')
                   and item not in ['__pycache__', '__MACOSX']]
        
        if not datasets:
            print("❌ No datasets found!")
            return
        
        print(f"Found {len(datasets)} dataset(s) to process:\n")
        
        for i, dataset in enumerate(datasets, 1):
            dataset_path = os.path.join(parent_path, dataset)
            print(f"[{i}/{len(datasets)}] Processing: {dataset}")
            
            # Set the global path variable
            path = dataset_path
            
            # Run the analysis (your existing code)
            print_tree(save_to_string=True)
            count_extensions(show_zip_contents=show_zip_contents)
            
            # Save output file
            base_name = os.path.splitext(dataset)[0]
            output_filename = f"directory_tree_{base_name}.txt"
            saved_path = save_tree_to_file(output_filename, tree_output, include_summary=True)
            
            print(f"✅ Completed: {dataset}\n")
        
        print(f"🎉 All {len(datasets)} datasets processed successfully!")
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")

FOR BATCH DATASET

In [ ]:
# Set your parent directory path containing multiple datasets
parent_directory = "/Volumes/PPS64/Datasets/UKCS"

# Process all datasets automatically
process_batch_datasets(parent_directory, show_zip_contents=True)

Visualisation

In [ ]:
# ==============================
# TREEMAP VISUALIZATION (SAVE ONLY)
# ==============================

import matplotlib.pyplot as plt
import squarify
import os

# ---------- Size formatter (same style as summary) ----------
def format_size_summary(size_bytes):
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} PB"


# ---------- Color palette ----------
# Carefully chosen readable, non-harsh professional colors
TREEMAP_COLORS = [
    "#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974",
    "#64B5CD", "#8C8C8C", "#E17C05", "#76B7B2", "#F28E2B"
]


# ---------- Core Treemap Plotter ----------
def save_treemap(extension_sizes, dataset_name):
    """
    Builds and saves treemap image for current dataset.
    Image is saved in same directory as your tree text output.
    Now also shows percentage contribution.
    """

    if not extension_sizes:
        print("❌ No extension data to plot.")
        return

    # Prepare data
    labels = []
    sizes = []

    for ext, size in extension_sizes.items():
        if size > 0:
            labels.append(ext.upper())
            sizes.append(size)

    if not sizes:
        print("❌ No non-zero file sizes found.")
        return

    # --- Compute total for percentage ---
    total_size = sum(sizes)

    # Labels with size + percentage
    display_labels = [
        f"{l}\n{format_size_summary(s)}\n({(s/total_size)*100:.1f}%)"
        for l, s in zip(labels, sizes)
    ]

    # Colors
    colors = (TREEMAP_COLORS * ((len(sizes) // len(TREEMAP_COLORS)) + 1))[:len(sizes)]

    # ---- Build figure ----
    fig = plt.figure(figsize=(14, 9))
    ax = fig.add_subplot(111)

    squarify.plot(
        sizes=sizes,
        label=display_labels,
        color=colors,
        alpha=0.88,
        text_kwargs={'fontsize':10, 'fontweight':'bold'},
        ax=ax
    )

    ax.set_title(
        f"Dataset File Composition\n{dataset_name}",
        fontsize=16,
        fontweight="bold"
    )

    ax.axis("off")
# ---- Save location (same as your text output folder) ----
    base_dir = os.path.dirname(path) if os.path.isfile(path) else os.path.dirname(path.rstrip(os.sep))
    image_filename = f"treemap_{dataset_name}.png"
    save_path = os.path.join(base_dir, image_filename)

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"📊 Treemap saved → {save_path}")


# =========================================
# SINGLE DATASET VISUALIZATION
# =========================================

def visualize_single_dataset():
    """
    Call AFTER:
        print_tree(...)
        count_extensions(...)
    """
    dataset_name = os.path.splitext(os.path.basename(path))[0]
    save_treemap(extension_sizes, dataset_name)


# =========================================
# BATCH DATASET VISUALIZATION
# =========================================

def visualize_batch_datasets(parent_path, show_zip_contents=True):
    """
    Runs your analyzer + saves treemap for each dataset
    """

    items = os.listdir(parent_path)

    datasets = [
        item for item in items
        if (os.path.isdir(os.path.join(parent_path, item))
            or item.lower().endswith('.zip'))
        and not item.startswith('.')
        and item not in ['__pycache__', '__MACOSX']
    ]

    if not datasets:
        print("❌ No datasets found.")
        return

    for dataset in datasets:
        print(f"\n📂 Processing dataset: {dataset}")

        global path
        path = os.path.join(parent_path, dataset)

        # --- Your existing analyzer calls ---
        print_tree(save_to_string=True)
        count_extensions(show_zip_contents=show_zip_contents)

        # --- Save treemap ---
        dataset_name = os.path.splitext(dataset)[0]
        save_treemap(extension_sizes, dataset_name)

    print("\n🎉 All treemap images saved successfully!")

In [ ]:
visualize_single_dataset()

In [ ]:
visualize_batch_datasets(parent_directory)